# 지역별고용조사 기반 청년층 정규직 근로자 비율 산출

2016–2024년 지역별고용조사 C형 상·하반기 자료를 각각 독립된 시점으로 사용해 시도별 청년층 정규직 근로자 비율을 산출한다.

- 원자료: [통계청 MDIS 지역별고용조사](https://mdis.mods.go.kr/ofrData/selectOfrDataDetail.do?itemId=2004&itemNm=%EB%85%B8%EB%8F%99&itmDiv=1&nPage=3&survId=1003790) C형 마이크로데이터
- 원자료 식별: 지역별고용조사 반기 C형 시도 중분류 부가항목, 2016–2024년 상·하반기 18개 CSV
- 재현성 자료: 연도별 파일설계서·코드북으로 컬럼명과 코드를 교차 확인하며, 원자료와 MDIS 다운로드 내역은 Git에 포함하지 않는다.
- 분모: 만 19–34세, 경제활동상태 코드 1, 종사상지위 코드 1·2
- 분자: 분모 중 종사상지위 코드 1이고 주업·부업 총계 근무시간 구분 코드 3
- 산식: 분자 가중치 합계 ÷ 분모 가중치 합계 × 100
- 가중치: 각 파일에 단일 제공된 시도 단위 가중치 컬럼을 사용한다. 현재 자료는 상반기 `시도전국가중값`, 하반기 `시도가중값`을 공통 `가중치`로 매핑한다.
- 파일별 시도 비율을 먼저 산출하고, 최종 출력에서만 소수점 첫째 자리로 반올림한다.


In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

np.random.seed(42)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = REPO_ROOT / "data/raw/구조환경지수 원데이터 구축용/지역별고용조사"
OUTPUT_DIR = REPO_ROOT / "data/processed/analysis"
OUTPUT_PATH = OUTPUT_DIR / "2016-2024_지역별고용조사_청년층_정규직_근로자_비율.csv"

SIDO_NAMES = {
    11: "서울",
    21: "부산",
    22: "대구",
    23: "인천",
    24: "광주",
    25: "대전",
    26: "울산",
    29: "세종",
    31: "경기",
    32: "강원",
    33: "충북",
    34: "충남",
    35: "전북",
    36: "전남",
    37: "경북",
    38: "경남",
    39: "제주",
}
HALF_ORDER = {"상반기": 0, "하반기": 1}

In [2]:
file_records = []
for path in RAW_DIR.glob("*.csv"):
    match = re.match(r"^(20\d{2})_(상반기|하반기)\(C형", path.name)
    if match:
        year, half = match.groups()
        file_records.append(
            {"연도": int(year), "반기": half, "시점": f"{year}_{half}", "경로": path}
        )

file_index = (
    pd.DataFrame(file_records)
    .assign(반기순서=lambda x: x["반기"].map(HALF_ORDER))
    .sort_values(["연도", "반기순서"])
    .drop(columns="반기순서")
    .reset_index(drop=True)
)
assert len(file_index) == 18
file_index.assign(
    경로=lambda x: x["경로"].map(lambda path: path.relative_to(REPO_ROOT).as_posix())
)[["연도", "반기", "시점", "경로"]]

,연도,반기,시점,경로
0,2016,상반기,2016_상반기,data/raw/구조환경지수 원데이터 구축용/지역별고용조사/...
1,2016,하반기,2016_하반기,data/raw/구조환경지수 원데이터 구축용/지역별고용조사/...
2,2017,상반기,2017_상반기,data/raw/구조환경지수 원데이터 구축용/지역별고용조사/...
3,2017,하반기,2017_하반기,data/raw/구조환경지수 원데이터 구축용/지역별고용조사/...
4,2018,상반기,2018_상반기,data/raw/구조환경지수 원데이터 구축용/지역별고용조사/...
5,2018,하반기,2018_하반기,data/raw/구조환경지수 원데이터 구축용/지역별고용조사/...
6,2019,상반기,2019_상반기,data/raw/구조환경지수 원데이터 구축용/지역별고용조사/...
7,2019,하반기,2019_하반기,data/raw/구조환경지수 원데이터 구축용/지역별고용조사/...
8,2020,상반기,2020_상반기,data/raw/구조환경지수 원데이터 구축용/지역별고용조사/...
9,2020,하반기,2020_하반기,data/raw/구조환경지수 원데이터 구축용/지역별고용조사/...


In [3]:
COLUMN_ALIASES = {
    "시도코드": ["행정구역시도코드", "2자리_행정구역시도코드"],
    "만연령": ["만연령"],
    "경제활동상태": ["경제활동인구상태코드", "경제활동구분코드"],
    "종사상지위": ["종사상지위코드", "현직장종사상지위코드"],
    "근무시간구분": ["주업부업총계시간구분코드"],
    "가중치": ["시도전국가중값", "시도가중값"],
}
COMMON_COLUMNS = list(COLUMN_ALIASES)

In [4]:
def calculate_sido_rate(path):
    data = pd.read_csv(path, encoding="cp949")
    rename_map = {}
    for common, aliases in COLUMN_ALIASES.items():
        matched_aliases = [alias for alias in aliases if alias in data.columns]
        if not matched_aliases:
            raise KeyError(
                f"{path.name}: '{common}'에 해당하는 컬럼을 찾지 못했습니다 (후보: {aliases})"
            )
        if len(matched_aliases) > 1:
            raise ValueError(
                f"{path.name}: '{common}' 후보가 둘 이상 존재합니다 "
                f"(매칭: {matched_aliases}). 사용할 컬럼을 명시적으로 확정해야 합니다."
            )
        rename_map[matched_aliases[0]] = common
    data = data.rename(columns=rename_map)[COMMON_COLUMNS]

    denominator = (
        data["만연령"].between(19, 34)
        & data["경제활동상태"].eq(1)
        & data["종사상지위"].isin([1, 2])
    )
    numerator = denominator & data["종사상지위"].eq(1) & data["근무시간구분"].eq(3)

    denominator_weight = data.loc[denominator].groupby("시도코드")["가중치"].sum()
    numerator_weight = (
        data.loc[numerator]
        .groupby("시도코드")["가중치"]
        .sum()
        .reindex(denominator_weight.index, fill_value=0)
    )
    return numerator_weight.div(denominator_weight).mul(100)


period_results = {
    row.시점: calculate_sido_rate(row.경로) for row in file_index.itertuples(index=False)
}
raw_result = pd.DataFrame(period_results).reindex(index=list(SIDO_NAMES))
raw_result.index.name = "시도코드"

In [5]:
final_df = raw_result.rename(index=SIDO_NAMES).round(1).reset_index(names="시도")
assert final_df.shape == (17, 19)
final_df

,시도,2016_상반기,2016_하반기,2017_상반기,2017_하반기,2018_상반기,2018_하반기,2019_상반기,2019_하반기,2020_상반기,2020_하반기,2021_상반기,2021_하반기,2022_상반기,2022_하반기,2023_상반기,2023_하반기,2024_상반기,2024_하반기
0,서울,58.5,65.1,68.3,64.7,67.6,65.4,67.6,69.3,45.4,67.9,69.5,52.5,70.6,51.5,74.6,72.5,72.3,71.7
1,부산,58.6,63.8,65.8,65.4,66.8,68.3,69.2,66.4,47.2,65.7,69.6,63.3,66.7,31.0,65.2,64.0,67.2,68.3
2,대구,60.4,63.2,68.0,66.2,63.8,63.6,67.0,65.2,53.1,66.2,66.4,61.1,69.3,56.6,70.6,69.5,72.5,72.4
3,인천,59.1,62.7,63.5,63.8,66.4,63.5,61.4,62.7,44.0,63.0,66.0,64.9,70.0,58.9,70.9,72.8,72.1,69.7
4,광주,64.6,65.8,64.8,62.6,62.6,63.0,66.7,65.4,53.4,62.6,65.5,63.5,67.3,58.7,68.8,66.5,68.3,68.7
5,대전,59.4,66.3,70.3,69.1,68.1,64.7,72.1,67.0,53.2,63.3,66.8,48.6,70.0,43.0,74.7,74.9,70.8,69.2
6,울산,68.3,73.1,71.3,70.6,69.4,67.9,72.7,68.4,53.2,66.8,76.6,63.9,73.8,63.9,73.5,73.7,70.6,68.9
7,세종,NaN,NaN,73.5,71.7,72.8,71.7,74.6,68.1,32.9,78.0,80.4,58.9,75.4,53.7,75.4,76.2,72.6,69.6
8,경기,61.7,68.5,69.3,67.9,69.4,69.5,68.0,67.9,45.3,68.9,70.1,56.5,70.1,50.3,72.9,70.8,70.6,69.7
9,강원,52.8,66.8,68.6,66.5,67.6,65.7,66.6,67.5,39.3,65.7,64.1,49.1,68.3,42.3,68.6,68.1,70.5,70.5


In [6]:
# 외부 기준값 출처:
# 강권오·황은진(2025), 『제주지역 출산환경지수 개발 연구』,
# <표 Ⅴ-1> 고용여건 지표 시도별 현황, p.87, 제주여성가족연구원(2025-11-15 발행).
published_jeju_2024_h1 = 65.4
jeju_2024_h1_raw = raw_result.loc[39, "2024_상반기"]
jeju_2024_h1_rounded = round(jeju_2024_h1_raw, 1)
jeju_2024_h2_raw = raw_result.loc[39, "2024_하반기"]
jeju_2024_h2_rounded = round(jeju_2024_h2_raw, 1)

jeju_comparison = pd.DataFrame(
    {
        "값": [
            published_jeju_2024_h1,
            jeju_2024_h1_raw,
            jeju_2024_h1_rounded,
            jeju_2024_h1_raw - published_jeju_2024_h1,
            jeju_2024_h1_rounded - published_jeju_2024_h1,
            jeju_2024_h2_raw,
            jeju_2024_h2_rounded,
        ]
    },
    index=[
        "2024년 상반기 외부 기준값",
        "2024년 상반기 계산값(반올림 전)",
        "2024년 상반기 계산값(소수점 첫째 자리)",
        "2024년 상반기 차이(반올림 전)",
        "2024년 상반기 차이(소수점 첫째 자리)",
        "2024년 하반기 계산값(반올림 전)",
        "2024년 하반기 계산값(소수점 첫째 자리)",
    ],
)
jeju_comparison

,값
2024년 상반기 외부 기준값,65.400000
2024년 상반기 계산값(반올림 전),65.477988
2024년 상반기 계산값(소수점 첫째 자리),65.500000
2024년 상반기 차이(반올림 전),0.077988
2024년 상반기 차이(소수점 첫째 자리),0.100000
2024년 하반기 계산값(반올림 전),63.589094
2024년 하반기 계산값(소수점 첫째 자리),63.600000


### 제주 외부 기준값 비교 결과

제주 2024년 상반기 외부 기준값 `65.4%`는 강권오·황은진(2025), 『제주지역 출산환경지수 개발 연구』의 **<표 Ⅴ-1> 고용여건 지표 시도별 현황(p.87)**에 제시된 측정값이다. 보고서는 제주여성가족연구원 기본연구 2025-04이며 2025년 11월 15일 발행되었다. [원문 PDF](https://drive.google.com/file/d/1puKooXi9ChZ_jrQRN2S1luCDQtJHOBOK)

종사상지위 코드 3인 비임금근로자를 분모에서 제외한 수정 산식의 2024년 상반기 제주 값은 반올림 전 65.4780%이고 소수점 첫째 자리 반올림값은 65.5%이다. 외부 기준값보다 0.1%p 높으며, 이 차이는 별도 원인 확인 대상으로 남긴다. 계산값을 외부 기준값에 맞춰 조정하지 않았고 비임금근로자 코드 3도 분모에 포함하지 않았다.


In [7]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
final_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

reloaded = pd.read_csv(OUTPUT_PATH, encoding="utf-8-sig")
expected_columns = ["시도", *file_index["시점"].tolist()]
assert reloaded.shape == (17, 19)
assert reloaded.columns.tolist() == expected_columns
assert reloaded["시도"].tolist() == list(SIDO_NAMES.values())
assert reloaded.isna().equals(final_df.isna())

print(f"저장 경로: {OUTPUT_PATH.relative_to(REPO_ROOT)}")
print(f"최종 크기: {reloaded.shape[0]}개 시도 × {reloaded.shape[1] - 1}개 반기 값")
print(f"2016년 상반기 결측 시도 수: {reloaded['2016_상반기'].isna().sum()}")

저장 경로: data/processed/analysis/2016-2024_지역별고용조사_청년층_정규직_근로자_비율.csv
최종 크기: 17개 시도 × 18개 반기 값
2016년 상반기 결측 시도 수: 1


## 반기 → 연도 집계 (연평균 채택)

이슈 #59 매니페스트 최신화 리포트 4절에서 미확정으로 남겼던 반기→연도
집계 방식을 팀 논의 결과 **연평균(상반기·하반기 단순평균)**으로
확정했다. 28개 통합 패널의 다른 지표와 동일하게 시도×연도 단위로
맞추기 위해, 반올림 전 비율(`raw_result`)을 기준으로 연평균을 계산한
뒤 한 번만 반올림한다(반기값과 연도값을 각각 반올림해 더하는 이중
반올림 오차를 피하기 위함).


In [8]:
YEARS = sorted({int(col.split("_")[0]) for col in raw_result.columns})

annual_raw = pd.DataFrame(
    {year: raw_result[[f"{year}_상반기", f"{year}_하반기"]].mean(axis=1, skipna=False) for year in YEARS},
    index=raw_result.index,
)
annual_df = annual_raw.rename(index=SIDO_NAMES).round(1).reset_index(names="시도")
annual_df.columns = ["시도", *[str(year) for year in YEARS]]
assert annual_df.shape == (17, len(YEARS) + 1)

ANNUAL_OUTPUT_PATH = OUTPUT_DIR / "2016-2024_지역별고용조사_청년층_정규직_근로자_비율_연도평균.csv"
annual_df.to_csv(ANNUAL_OUTPUT_PATH, index=False, encoding="utf-8-sig")

reloaded_annual = pd.read_csv(ANNUAL_OUTPUT_PATH, encoding="utf-8-sig")
expected_annual_columns = ["시도", *[str(year) for year in YEARS]]
assert reloaded_annual.shape == (17, len(YEARS) + 1)
assert reloaded_annual.columns.tolist() == expected_annual_columns
assert reloaded_annual["시도"].tolist() == list(SIDO_NAMES.values())
pd.testing.assert_frame_equal(
    reloaded_annual,
    annual_df,
    check_dtype=False,
    check_exact=True,
)

print(f"저장 경로: {ANNUAL_OUTPUT_PATH.relative_to(REPO_ROOT)}")
print(f"최종 크기: {reloaded_annual.shape[0]}개 시도 x {reloaded_annual.shape[1] - 1}개 연도 값")
annual_df

저장 경로: data/processed/analysis/2016-2024_지역별고용조사_청년층_정규직_근로자_비율_연도평균.csv
최종 크기: 17개 시도 x 9개 연도 값


,시도,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,서울,61.8,66.5,66.5,68.5,56.6,61.0,61.0,73.6,72.0
1,부산,61.2,65.6,67.5,67.8,56.5,66.4,48.9,64.6,67.7
2,대구,61.8,67.1,63.7,66.1,59.6,63.8,62.9,70.1,72.5
3,인천,60.9,63.7,65.0,62.0,53.5,65.4,64.4,71.9,70.9
4,광주,65.2,63.7,62.8,66.1,58.0,64.5,63.0,67.6,68.5
5,대전,62.9,69.7,66.4,69.5,58.2,57.7,56.5,74.8,70.0
6,울산,70.7,70.9,68.7,70.5,60.0,70.2,68.9,73.6,69.7
7,세종,NaN,72.6,72.3,71.3,55.4,69.7,64.6,75.8,71.1
8,경기,65.1,68.6,69.5,67.9,57.1,63.3,60.2,71.8,70.2
9,강원,59.8,67.5,66.7,67.1,52.5,56.6,55.3,68.3,70.5


## 내부 재현 일치 검증과 외부 기준 대조

아래 검증은 산출 함수 자체를 호출하지 않고 원자료를 다시 읽어 동일 조건으로 재계산한다. 따라서 코드 경로의 재현성과 최종 결과 일치는 확인하지만, 같은 컬럼 정의·산식·가중치 선택을 사용하므로 그 정의 자체가 잘못된 경우까지 발견하는 완전한 독립 검증은 아니다.

- **내부 검증:** 2019년 상반기와 2024년 상반기의 별도 재계산값이 최종 결과와 일치하는지 확인
- **외부 기준 대조:** 제주 2024년 상반기 값을 제주여성가족연구원 보고서의 65.4%와 별도로 비교


In [9]:
# 검증 전용: 기존 calculate_sido_rate 함수를 호출하지 않고 동일 원자료를 별도 코드 경로로 다시 계산해 내부 재현 일치를 확인한다.
verification_specs = [
    {
        "구분": "구형 파일 내부 재현 일치",
        "시점": "2019_상반기",
        "경로": file_index.loc[file_index["시점"].eq("2019_상반기"), "경로"].iloc[0],
        "시도": "행정구역시도코드",
        "연령": "만연령",
        "경제활동": "경제활동인구상태코드",
        "종사상지위": "종사상지위코드",
        "근무시간": "주업부업총계시간구분코드",
        "가중치": "시도전국가중값",
    },
    {
        "구분": "신형 파일 내부 재현 일치",
        "시점": "2024_상반기",
        "경로": file_index.loc[file_index["시점"].eq("2024_상반기"), "경로"].iloc[0],
        "시도": "행정구역시도코드",
        "연령": "만연령",
        "경제활동": "경제활동인구상태코드",
        "종사상지위": "종사상지위코드",
        "근무시간": "주업부업총계시간구분코드",
        "가중치": "시도전국가중값",
    },
]

final_values = final_df.set_index("시도")
reproduction_pass = {}
mismatch_tables = []

for spec in verification_specs:
    source = pd.read_csv(spec["경로"], encoding="cp949")
    denominator = (
        source[spec["연령"]].between(19, 34)
        & source[spec["경제활동"]].eq(1)
        & source[spec["종사상지위"]].isin([1, 2])
    )
    numerator = denominator & source[spec["종사상지위"]].eq(1) & source[spec["근무시간"]].eq(3)
    denominator_weight = source.loc[denominator].groupby(spec["시도"])[spec["가중치"]].sum()
    numerator_weight = source.loc[numerator].groupby(spec["시도"])[spec["가중치"]].sum()
    check = pd.concat(
        [denominator_weight.rename("분모가중치합계"), numerator_weight.rename("분자가중치합계")],
        axis=1,
    ).reindex(SIDO_NAMES)
    check["재계산값"] = check["분자가중치합계"].div(check["분모가중치합계"]).mul(100)
    check["지역"] = check.index.map(SIDO_NAMES)
    check["최종값"] = check["지역"].map(final_values[spec["시점"]])
    check["차이"] = check["재계산값"].round(1) - check["최종값"]
    check["일치"] = np.isclose(check["재계산값"].round(1), check["최종값"], equal_nan=True)
    reproduction_pass[spec["구분"]] = bool(check["일치"].all())
    mismatch = check.loc[~check["일치"], ["지역", "재계산값", "최종값", "차이"]].copy()
    if not mismatch.empty:
        mismatch.insert(0, "시점", spec["시점"])
        mismatch_tables.append(mismatch.reset_index(drop=True))

non_missing_values = final_values.stack().dropna()
range_pass = bool(non_missing_values.between(0, 100).all())
missing_rows = [
    {"지역": region, "시점": period}
    for region in final_values.index
    for period in final_values.columns
    if pd.isna(final_values.loc[region, period])
]
missing_df = pd.DataFrame(missing_rows, columns=["지역", "시점"])
expected_missing = {("세종", "2016_상반기"), ("세종", "2016_하반기")}
actual_missing = set(map(tuple, missing_df[["지역", "시점"]].to_numpy()))
unexpected_missing_pass = not bool(actual_missing - expected_missing)

previous_values = final_values.shift(axis=1)
changes = (final_values - previous_values).round(1)
rapid_rows = []
for column_number, period in enumerate(final_values.columns[1:], start=1):
    for region in final_values.index[changes[period].abs().gt(10)]:
        rapid_rows.append(
            {
                "지역": region,
                "시점": period,
                "이전시점": final_values.columns[column_number - 1],
                "이전값": previous_values.loc[region, period],
                "현재값": final_values.loc[region, period],
                "증감폭": changes.loc[region, period],
            }
        )
rapid_df = pd.DataFrame(
    rapid_rows, columns=["지역", "시점", "이전시점", "이전값", "현재값", "증감폭"]
)

sejong_rows = []
for period, sido_column in [
    ("2016_상반기", "행정구역시도코드"),
    ("2016_하반기", "2자리_행정구역시도코드"),
]:
    path = file_index.loc[file_index["시점"].eq(period), "경로"].iloc[0]
    sido_codes = sorted(
        pd.read_csv(path, encoding="cp949", usecols=[sido_column])[sido_column]
        .dropna()
        .unique()
        .tolist()
    )
    has_29 = 29 in sido_codes
    result_missing = pd.isna(final_values.loc["세종", period])
    sejong_rows.append(
        {
            "시점": period,
            "원자료 시도코드 고유값": sido_codes,
            "코드29 존재": has_29,
            "결과 결측": result_missing,
        }
    )
sejong_df = pd.DataFrame(sejong_rows)
sejong_absence_pass = bool((~sejong_df["코드29 존재"] & sejong_df["결과 결측"]).all())

csv_check = pd.read_csv(OUTPUT_PATH, encoding="utf-8-sig")
csv_structure_pass = bool(
    csv_check.shape == (17, 19)
    and csv_check.columns.tolist() == final_df.columns.tolist()
    and csv_check["시도"].tolist() == final_df["시도"].tolist()
)
try:
    pd.testing.assert_frame_equal(csv_check, final_df, check_dtype=False, check_exact=True)
    csv_values_pass = True
except AssertionError:
    csv_values_pass = False

summary = {
    "구형 파일 내부 재현 일치": reproduction_pass["구형 파일 내부 재현 일치"],
    "신형 파일 내부 재현 일치": reproduction_pass["신형 파일 내부 재현 일치"],
    "값 범위": range_pass,
    "예상 외 결측": unexpected_missing_pass,
    "세종 원자료 부재": sejong_absence_pass,
    "CSV 구조": csv_structure_pass,
    "CSV 값 일치": csv_values_pass,
}
summary_df = pd.DataFrame(
    {
        "검증 항목": summary.keys(),
        "결과": ["PASS" if value else "FAIL" for value in summary.values()],
    }
)

print("[내부 재현 불일치 지역]")
display(pd.concat(mismatch_tables, ignore_index=True) if mismatch_tables else "없음")
print("[전체 결측 위치]")
display(missing_df if not missing_df.empty else "없음")
print("[2016년 세종 원자료 확인]")
for row in sejong_rows:
    print(
        f"{row['시점']} 고유값: {row['원자료 시도코드 고유값']}, 코드29 존재: {row['코드29 존재']}, 결과 결측: {row['결과 결측']}"
    )
print("[최종 검증 요약]")
display(summary_df)
print("[검토 필요 항목: 직전 반기 대비 절대 증감 10%p 초과]")
with pd.option_context("display.max_rows", None):
    display(rapid_df if not rapid_df.empty else "없음")

[내부 재현 불일치 지역]


'없음'

[전체 결측 위치]


,지역,시점
0,세종,2016_상반기
1,세종,2016_하반기


[2016년 세종 원자료 확인]
2016_상반기 고유값: [11, 21, 22, 23, 24, 25, 26, 31, 32, 33, 34, 35, 36, 37, 38, 39], 코드29 존재: False, 결과 결측: True
2016_하반기 고유값: [11, 21, 22, 23, 24, 25, 26, 31, 32, 33, 34, 35, 36, 37, 38, 39], 코드29 존재: False, 결과 결측: True
[최종 검증 요약]


,검증 항목,결과
0,구형 파일 내부 재현 일치,PASS
1,신형 파일 내부 재현 일치,PASS
2,값 범위,PASS
3,예상 외 결측,PASS
4,세종 원자료 부재,PASS
5,CSV 구조,PASS
6,CSV 값 일치,PASS


[검토 필요 항목: 직전 반기 대비 절대 증감 10%p 초과]


,지역,시점,이전시점,이전값,현재값,증감폭
0,강원,2016_하반기,2016_상반기,52.8,66.8,14.0
1,서울,2020_상반기,2019_하반기,69.3,45.4,-23.9
2,부산,2020_상반기,2019_하반기,66.4,47.2,-19.2
3,대구,2020_상반기,2019_하반기,65.2,53.1,-12.1
4,인천,2020_상반기,2019_하반기,62.7,44.0,-18.7
5,광주,2020_상반기,2019_하반기,65.4,53.4,-12.0
6,대전,2020_상반기,2019_하반기,67.0,53.2,-13.8
7,울산,2020_상반기,2019_하반기,68.4,53.2,-15.2
8,세종,2020_상반기,2019_하반기,68.1,32.9,-35.2
9,경기,2020_상반기,2019_하반기,67.9,45.3,-22.6


## 해석상 한계

- 이 지표는 조사대상주간의 실제 주업·부업 총 취업시간 36시간 이상 여부를 사용한다.
- 2020년 상반기, 2021년 하반기, 2022년 하반기처럼 조사대상주간에 선거일 또는 대체공휴일이 포함된 시점은 36시간 이상 근로자 비율이 기계적으로 낮아질 수 있다.
- 따라서 급변값은 산출 오류로 판정하지 않되, 인접 반기 간 증감을 순수한 고용 안정성 변화로 해석하지 않는다.
- 향후 시계열 분석에서는 동일 반기끼리 비교하거나 조사대상주간 효과를 별도로 고려해야 한다.
- 2020년 상반기는 코로나19 영향도 함께 고려해야 한다.
- 세종·제주처럼 분모 표본 수가 작은 시도는 반기별 추정치의 변동성과 불확실성이 커질 수 있다. 급변값은 비가중 분모 표본 수와 함께 해석해야 한다.
- 현재 결과는 복합표본설계를 반영한 표준오차·신뢰구간을 제시하지 않으므로, 지역 간 미세한 차이를 확정적 순위 차이로 해석하지 않는다.
- 현재 18개 C형 원자료에는 파일마다 시도 단위 가중치 컬럼이 하나만 존재하며, 상반기는 `시도전국가중값`, 하반기는 `시도가중값`으로 제공된다. 2019년 상반기와 2024년 하반기 파일설계서에서도 각각 해당 컬럼이 단일 가중치 항목으로 확인된다.
- 두 이름은 파일별 컬럼명 차이로 취급해 공통 `가중치`로 매핑한다. 향후 한 파일에 복수 가중치 후보가 동시에 존재하면 자동 선택하지 않고 오류를 발생시켜 선택 근거를 다시 확인한다.
